# Part B - Low-Rank Adaptation (LoRA)
## TOTAL MARKS: 50
## Overview
### `Name: Muhammad Bilal Hassan`
### `Roll Number: 28100371`

References:
https://magazine.sebastianraschka.com/p/lora-and-dora-from-scratch



# Low-Rank Adaptation (LoRA) — An Intuitive Explanation

Large language models (LLMs) and vision transformers contain **millions or billions of parameters** (weights).  
Fine-tuning these models normally requires updating *all* of these weights, which is computationally expensive and often limited by GPU memory.

---

### Regular Fine-Tuning

In standard training or fine-tuning:

- Each layer has a large weight matrix **W**
- Training learns a full update matrix **ΔW**
- The updated weights are:

$$
W_{\text{updated}} = W + \Delta W
$$

**Drawback:**  
The update matrix **ΔW** is very large, making training slow and memory-intensive.

---

### LoRA: The Key Idea

LoRA (Low-Rank Adaptation) is based on the observation that:

> Most useful weight updates can be represented by **simple patterns**, rather than a full, complex matrix.

Instead of learning the full **ΔW**, LoRA **approximates it** using two much smaller matrices:

$$
\Delta W \approx A \cdot B
$$

where:
- **A** and **B** are small matrices
- Their matrix product has the same shape as **W**

The weight update becomes:

$$
W_{\text{updated}} = W + A \cdot B
$$

The figure below illustrates these formulas for full finetuning and LoRA side by side.

![LoRA illustration](LoRAImage.webp)

---

### Intuitive Explanation

Think of the model as a large machine with many adjustment knobs.

- **Full fine-tuning:**  
  Adjust every knob individually.

- **LoRA:**  
  Keep the original knobs fixed and add a small attachment that adjusts many knobs together in a coordinated way.

This attachment (matrices **A** and **B**) captures the most important changes while using far fewer parameters.

## Implementing a LoRA Layer (2.5 MARKS)
Implement a LoRA layer that adapts a neural network without directly modifying its original weights. Instead of learning a full weight update, represent the adaptation as a low-rank decomposition that captures task-specific changes using a compact set of parameters. This formulation reduces both memory usage and computational cost compared to full fine-tuning.

Design the LoRA layer using two trainable low-rank matrices whose product defines the weight update. Scale this update appropriately and optionally apply dropout to regulate its effect. During the forward pass, compute the low-rank update independently so it can later be combined with the output of a frozen base layer in a modular and efficient manner.

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [22]:
class LoRALayer(nn.Module):
    """
    Implements a LoRA low-rank adaptation layer.
    """

    def __init__(self, in_features, out_features, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        # TODO: store rank and alpha
        self.rank = rank
        self.alpha = alpha
        # TODO: initialize A and B matrices
        self.A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, out_features))
        # TODO: initialize dropout
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        # TODO: apply dropout
        x = self.dropout(x)
        # TODO: compute low-rank update
        low_rank_update = (x @ self.A) @ self.B
        # TODO: scale and return
        return low_rank_update * (self.alpha / self.rank)


## LoRA-Adapted Linear Layer (non-merged) (5 MARKS)
Implement a LoRA-adapted version of a linear layer by wrapping an existing nn.Linear module. The original weights and bias should be frozen to ensure that only the low-rank adaptation is trained. The forward pass should combine the output of the frozen linear layer with the output of the LoRA layer, illustrating how task-specific updates can be added without modifying the base parameters.

In [23]:
class LoRALinear(nn.Module):
    """
    A linear layer with LoRA adaptation.
    """

    def __init__(self, linear_layer, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        # TODO: store linear layer
        self.linear_layer = linear_layer
        # TODO: freeze base weights
        for param in self.linear_layer.parameters():
            param.requires_grad = False
        # TODO: create LoRALayer
        self.lora_layer = LoRALayer(
            in_features=linear_layer.in_features,
            out_features=linear_layer.out_features,
            rank=rank,
            alpha=alpha,
            dropout=dropout
        )

    def forward(self, x):
        # TODO: compute base output
        base_output = self.linear_layer(x)
        # TODO: compute LoRA output
        lora_output = self.lora_layer(x)
        # TODO: return combined output
        return base_output + lora_output

## Define a Small Network with One Linear Layer (2.5 MARKS)
Create a simple experimental setup using a single linear layer and a randomly generated input tensor. Fix the random seed to ensure reproducibility. Compute and record the output of the original linear layer, which will serve as a baseline for comparison when LoRA is introduced.

In [24]:
# TODO: set random seed
random_seed = 64
torch.manual_seed(random_seed)

# TODO: create linear layer
in_features = 512
out_features = 512
linear_layer = nn.Linear(in_features, out_features)
# TODO: create input tensor
batch_size = 4
input_tensor = torch.randn(batch_size, in_features)
# TODO: compute and print baseline output
baseline_output = linear_layer(input_tensor)
print("Baseline output:")
print(baseline_output)

Baseline output:
tensor([[-0.2213,  1.2516,  0.3799,  ...,  0.4726, -0.3715,  0.8355],
        [ 0.2617,  0.5843, -0.5208,  ..., -0.1679, -0.2559, -0.6548],
        [ 0.2189, -1.0069, -1.3101,  ..., -0.9824, -0.5336,  0.5009],
        [ 0.1982, -0.8633, -0.3625,  ...,  0.6457, -0.5043, -0.1577]],
       grad_fn=<AddmmBackward0>)


## Apply LoRA to the Linear Layer (2.5 MARKS)
Apply the LoRA-adapted linear wrapper to the previously defined layer. Run the same input through this modified layer and observe the change in output compared to the baseline. This step demonstrates how LoRA introduces an additional learned transformation while keeping the original layer weights unchanged.

In [25]:
# TODO: wrap layer with LoRALinear
lora_linear_layer = LoRALinear(linear_layer, rank=2, alpha=4)
# TODO: compute and print LoRA output
lora_output = lora_linear_layer(input_tensor)
print("LoRA output:")
print(lora_output)

LoRA output:
tensor([[-0.2213,  1.2516,  0.3799,  ...,  0.4726, -0.3715,  0.8355],
        [ 0.2617,  0.5843, -0.5208,  ..., -0.1679, -0.2559, -0.6548],
        [ 0.2189, -1.0069, -1.3101,  ..., -0.9824, -0.5336,  0.5009],
        [ 0.1982, -0.8633, -0.3625,  ...,  0.6457, -0.5043, -0.1577]],
       grad_fn=<AddBackward0>)


## Merged LoRA Version (for inference) (5 MARKS)
Implement an alternative version of the linear layer in which the LoRA weight update is merged directly into the original weight matrix. In this formulation, compute the low-rank weight update and add it to the frozen base weights before applying the linear operation. This merged approach reflects how LoRA can be deployed efficiently during inference.

In [26]:
class LinearWithLoRAMerged(nn.Module):
    """
    Linear layer where LoRA weights are merged into W.
    """

    def __init__(self, linear, rank, alpha):
        super().__init__()
        # TODO: store linear layer
        self.linear_layer = linear

        # ---------
        for param in self.linear_layer.parameters():
            param.requires_grad = False
        # ---------
        # TODO: create LoRALayer
        self.lora_layer = LoRALayer(
            in_features=linear.in_features,
            out_features=linear.out_features,
            rank=rank,
            alpha=alpha,
            dropout=0.0
        )

    def forward(self, x):
        # TODO: compute delta W
        A = self.lora_layer.A
        B = self.lora_layer.B
        scaling_factor = self.lora_layer.alpha / self.lora_layer.rank
        delta_W = (A @ B).T * scaling_factor
        # TODO: merge weights
        merged_weights = self.linear_layer.weight + delta_W
        # TODO: apply linear op
        return F.linear(x, merged_weights, self.linear_layer.bias)


## Verify Mathematical Equivalence  (0.5 MARKS)
Verify that the merged and non-merged LoRA implementations produce identical outputs when initialized with the same parameters. Ensure both models share the same low-rank matrices, then compare their outputs on the same input. Report the maximum absolute difference to confirm numerical equivalence between the two approaches.

In [27]:
# TODO: instantiate merged and unmerged models
linear_layer_unmerged = LoRALinear(linear_layer, rank=2, alpha=4)
linear_layer_merged = LinearWithLoRAMerged(linear_layer, rank=2, alpha=4)
# TODO: copy LoRA parameters
with torch.no_grad():
    linear_layer_merged.lora_layer.A.copy_(linear_layer_unmerged.lora_layer.A)
    linear_layer_merged.lora_layer.B.copy_(linear_layer_unmerged.lora_layer.B)
# TODO: compare outputs
layer_lora_unmerged = linear_layer_unmerged(input_tensor)
layer_lora_merged = linear_layer_merged(input_tensor)
print("Unmerged LoRA output:")
print(layer_lora_unmerged)
print("Merged LoRA output:")
print(layer_lora_merged)

max_diff = torch.max(torch.abs(layer_lora_unmerged - layer_lora_merged)).item()
print(f"Max absolute difference between unmerged and merged outputs:{max_diff}")

if max_diff < 1e-5:
    print("Verification passed.")
else:
    print("Verification failed: There is a significant difference.")

Unmerged LoRA output:
tensor([[-0.2213,  1.2516,  0.3799,  ...,  0.4726, -0.3715,  0.8355],
        [ 0.2617,  0.5843, -0.5208,  ..., -0.1679, -0.2559, -0.6548],
        [ 0.2189, -1.0069, -1.3101,  ..., -0.9824, -0.5336,  0.5009],
        [ 0.1982, -0.8633, -0.3625,  ...,  0.6457, -0.5043, -0.1577]],
       grad_fn=<AddBackward0>)
Merged LoRA output:
tensor([[-0.2213,  1.2516,  0.3799,  ...,  0.4726, -0.3715,  0.8355],
        [ 0.2617,  0.5843, -0.5208,  ..., -0.1679, -0.2559, -0.6548],
        [ 0.2189, -1.0069, -1.3101,  ..., -0.9824, -0.5336,  0.5009],
        [ 0.1982, -0.8633, -0.3625,  ...,  0.6457, -0.5043, -0.1577]],
       grad_fn=<AddmmBackward0>)
Max absolute difference between unmerged and merged outputs:0.0
Verification passed.


# Applying LoRA Layers to LLM (2.5 MARKS)


## Preparing the Dataset
Prepare a text dataset suitable for causal language modeling. Load a subset of the AG News dataset, shuffle it for reproducibility, and split it into training and evaluation sets. Tokenize the text with fixed-length padding and truncation, and construct labels that mask padding tokens so they do not contribute to the training loss. Finally, format the dataset for PyTorch and create data loaders for batch-based training and evaluation.

In [28]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset


In [29]:
from datasets import load_dataset
from torch.utils.data import DataLoader

def prepare_dataset(tokenizer, max_samples=500, max_length=128):
    """
    Prepares train and test DataLoaders for language model fine-tuning.

    Args:
        tokenizer: HuggingFace tokenizer
        max_samples: number of training samples to use
        max_length: maximum sequence length

    Returns:
        train_loader, test_loader
    """
    dataset = load_dataset("ag_news")
    train_data = dataset['train'].shuffle(seed=42).select(range(max_samples))
    eval_samples = min(int(max_samples * 0.2), len(dataset['test']))
    test_data = dataset['test'].shuffle(seed=42).select(range(eval_samples))
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def tokenize_function(examples):
        tokens = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=max_length)
        inputs = tokens['input_ids']
        labels = []
        for seq in inputs:
            seq_labels = [-100 if token == tokenizer.pad_token_id else token for token in seq]
            labels.append(seq_labels)
        tokens['labels'] = labels
        return tokens
    
    tokenized_train = train_data.map(tokenize_function, batched=True, remove_columns=['text', 'label'])
    tokenized_test = test_data.map(tokenize_function, batched=True, remove_columns=['text', 'label'])
    tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
 
    train_loader = DataLoader(tokenized_train, batch_size=8, shuffle=True)
    test_loader = DataLoader(tokenized_test, batch_size=8, shuffle=False)

    return train_loader, test_loader

## Initializing the Base Language Model
Load a pre-trained causal language model and its corresponding tokenizer. Configure the tokenizer to use a valid padding token and move the model to the appropriate computation device. Keep this base model in evaluation mode to serve as a frozen reference for comparison with the LoRA-adapted version.

In [30]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [31]:
model_name = "EleutherAI/gpt-neo-125M"
# TODO: load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# TODO: load base model

base_model = AutoModelForCausalLM.from_pretrained(model_name, pad_token_id=tokenizer.eos_token_id)

# TODO: set device and eval mode
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model.to(device)
base_model.eval()
for param in base_model.parameters():
    param.requires_grad = False

print(f"Model successfully loaded and frozen on: {device}")

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model successfully loaded and frozen on: cpu


## Creating a LoRA-Adapted Model (2.5 MARKS)
Instantiate a second copy of the pre-trained language model that will be modified using LoRA. This model will be used for training and should be set to training mode. Maintaining separate base and LoRA-adapted models allows direct comparison of parameter counts and downstream performance.

In [32]:
# TODO: load LoRA model
model_lora = AutoModelForCausalLM.from_pretrained(model_name, pad_token_id=tokenizer.eos_token_id)
# TODO: set training mode
model_lora.to(device)
model_lora.train()
print("LoRA model loaded and set to training mode.")

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LoRA model loaded and set to training mode.


## Applying LoRA to Attention Layers (2.5 MARKS)
Modify the language model by replacing selected linear layers within the attention mechanism with LoRA-adapted linear layers. Target projection layers commonly used in self-attention, such as query, key, value, and output projections. This step injects low-rank adaptations into the model while leaving the original weights structurally intact.

In [33]:
def apply_lora_to_model(model, rank=8, alpha=16):
     """
    Replaces selected attention projection Linear layers with LoRALinear wrappers.

    Args:
        model: a transformer model (HuggingFace or similar)
        rank: LoRA rank r
        alpha: LoRA scaling alpha
        dropout: LoRA dropout (applied inside LoRALayer)

    Returns:
        model (modified in-place or returned for convenience)
    """
     target_layer = ["q_proj", "k_proj", "v_proj", "out_proj"]
     module_dict = dict(model.named_modules())
    # TODO: iterate modules
    # TODO: replace attention linear layers
     for name, module in list(model.named_modules()):
        if any(target in name for target in target_layer):
            if isinstance(module, nn.Linear):
                parent_module_name = name.rsplit('.', 1)[0]
                parent_module = module_dict[parent_module_name] 
                child_module_name = name.rsplit('.', 1)[-1]
                setattr(parent_module, child_module_name, LoRALinear(module, rank=rank, alpha=alpha))

     return model


In [34]:
model_lora = apply_lora_to_model(model_lora, rank=8, alpha=16)
print("LoRA layers applied to model.")

LoRA layers applied to model.


## Freezing Base Parameters and Enabling LoRA Training (2.5 MARKS)
Freeze all original model parameters to prevent them from being updated during training. Enable gradient updates only for the LoRA low-rank matrices. This ensures that learning is restricted to the parameter-efficient LoRA components while the pre-trained knowledge of the base model is preserved.

In [35]:
# TODO: freeze base parameters
for param in base_model.parameters():
    param.requires_grad = False
# TODO: enable LoRA parameters
for name, param in model_lora.named_parameters():
    if ".A" in name or ".B" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False
print("Base parameters frozen, LoRA parameters enabled.")


Base parameters frozen, LoRA parameters enabled.


## Inspecting Trainable Parameters
Verify that only the intended LoRA parameters are trainable by inspecting the model’s parameter list. This step helps confirm that the parameter-freezing strategy has been applied correctly before training begins.

In [36]:
def check_trainable_params(model):
    # TODO: print trainable parameters
    print("Trainable parameters:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"{name}: {param.shape}")

check_trainable_params(model_lora)

Trainable parameters:
transformer.h.0.attn.attention.k_proj.lora_layer.A: torch.Size([768, 8])
transformer.h.0.attn.attention.k_proj.lora_layer.B: torch.Size([8, 768])
transformer.h.0.attn.attention.v_proj.lora_layer.A: torch.Size([768, 8])
transformer.h.0.attn.attention.v_proj.lora_layer.B: torch.Size([8, 768])
transformer.h.0.attn.attention.q_proj.lora_layer.A: torch.Size([768, 8])
transformer.h.0.attn.attention.q_proj.lora_layer.B: torch.Size([8, 768])
transformer.h.0.attn.attention.out_proj.lora_layer.A: torch.Size([768, 8])
transformer.h.0.attn.attention.out_proj.lora_layer.B: torch.Size([8, 768])
transformer.h.1.attn.attention.k_proj.lora_layer.A: torch.Size([768, 8])
transformer.h.1.attn.attention.k_proj.lora_layer.B: torch.Size([8, 768])
transformer.h.1.attn.attention.v_proj.lora_layer.A: torch.Size([768, 8])
transformer.h.1.attn.attention.v_proj.lora_layer.B: torch.Size([8, 768])
transformer.h.1.attn.attention.q_proj.lora_layer.A: torch.Size([768, 8])
transformer.h.1.attn.atte

## Comparing Parameter Counts  (2.5 MARKS)
Compute and compare the total number of parameters and the number of trainable parameters for both the base model and the LoRA-adapted model. Report the percentage reduction in trainable parameters achieved through LoRA to quantify its efficiency benefits.

In [37]:
def count_parameters(model):
    # TODO: compute parameter counts
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

base_total_params, base_trainable_params = count_parameters(base_model)
lora_total_params, lora_trainable_params = count_parameters(model_lora)

lora_percent = (lora_trainable_params / lora_total_params) * 100 if lora_total_params > 0 else 0
reduction_percent = ((base_total_params - lora_trainable_params) / base_total_params) * 100 if base_total_params > 0 else 0
print(f"Base parameters: {base_total_params}")
print(f"Base trainable parameters: {base_trainable_params}")
print(f"LoRA total parameters: {lora_total_params}")
print(f"LoRA trainable parameters: {lora_trainable_params}")
print(f"LoRA trainable percentage: {lora_percent:.2f}%")
print(f"Total Reduction in Trainable: {reduction_percent:.2f}%")

Base parameters: 125198592
Base trainable parameters: 0
LoRA total parameters: 125788416
LoRA trainable parameters: 589824
LoRA trainable percentage: 0.47%
Total Reduction in Trainable: 99.53%


## Training the LoRA-Adapted Model  (2.5 MARKS)
Train the LoRA-adapted model using the prepared dataset and an appropriate optimizer. Perform multiple training epochs while monitoring the training loss. Since only a small subset of parameters is being updated, this training process is significantly more efficient than full fine-tuning.

In [38]:
# TODO: training loop
train_loader, test_loader = prepare_dataset(tokenizer, max_samples=500, max_length=128)
optimizer = torch.optim.AdamW(
    [p for p in model_lora.parameters() if p.requires_grad], 
    lr=5e-5)

model_lora.train()

for epoch in range(2):
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model_lora(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels']
            )
        loss = outputs.loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} completed. Average Loss: {avg_loss:.4f}")

## DO NOT FORGET TO PRINT YOUR LOSS

Epoch 1 completed. Average Loss: 4.3537
Epoch 2 completed. Average Loss: 4.2283


## Evaluating Model Accuracy  (5 MARKS)
Evaluate both the original base model and the LoRA-adapted model on the test dataset. Compute token-level accuracy while ignoring masked positions. Compare the results to assess how well the LoRA-based fine-tuning performs relative to the frozen pre-trained model.

In [39]:
@torch.no_grad()
def compute_accuracy(model, dataloader, device):
    """
    TODO:
    1. Disable gradients and set model to eval mode
    2. Get model logits for each batch
    3. Compute predictions using argmax
    4. Mask out padding tokens (-100)
    5. Return token-level accuracy (%)
    """
    model.eval()
    total_tokens = 0
    correct_predictions = 0
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = batch['labels'][..., 1:].contiguous()
        predictions = torch.argmax(shift_logits, dim=-1)
        mask = shift_labels != -100
        correct_predictions += (predictions[mask] == shift_labels[mask]).sum().item()
        total_tokens += mask.sum().item()
    accuracy = (correct_predictions / total_tokens) * 100 if total_tokens > 0 else 0
    return accuracy

base_train_accuracy = compute_accuracy(base_model, train_loader, DEVICE)
base_test_accuracy = compute_accuracy(base_model, test_loader, DEVICE)

lora_train_accuracy = compute_accuracy(model_lora, train_loader, DEVICE)
lora_test_accuracy = compute_accuracy(model_lora, test_loader, DEVICE)

print(f"Base Training Accuracy: {base_train_accuracy:.2f}%")
print(f"Base Test Accuracy: {base_test_accuracy:.2f}%")
print(f"Lora Training Accuracy: {lora_train_accuracy:.2f}%")
print(f"Lora Test Accuracy: {lora_test_accuracy:.2f}%")
print(f"Improvement in Test Accuracy: {lora_test_accuracy - base_test_accuracy:.2f}%")

Base Training Accuracy: 30.38%
Base Test Accuracy: 30.47%
Lora Training Accuracy: 32.60%
Lora Test Accuracy: 32.34%
Improvement in Test Accuracy: 1.88%


In [40]:
print(f"Test accuracy orig model: {compute_accuracy(base_model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")

Test accuracy orig model: 30.47%
Test accuracy LoRA model: 32.34%


# Reflective Questions (12 Marks)
Answer the reflective questions provided below.

Q1. Why is representing weight updates as a low-rank decomposition effective for large language models? 

Ans. According to research by Aghajanyan et al., massive, over-parameterized language models possess a remarkably low intrinsic dimension. Because of this, the weight updates ($\Delta W$) required for downstream task adaptation also exist within a low-dimensional subspace. Therefore, instead of calculating updates for the entire full-rank base model, we can represent these updates as a low-rank decomposition and restrict our training to this essential low-dimensional subspace. These lower dimensions capture the vast majority of the required task-specific transformations, which significantly reduces the parameter count. This saves immense computational resources and enables us to fine-tune the model highly efficiently.

Q2. What assumptions does LoRA make about the nature of task-specific adaptations?

Ans. The core assumption LoRA makes is that a large, pre-trained base model already contains a vast amount of general knowledge and representational power. Because of this, it assumes that new 'task-specific adaptations' do not require fundamentally rewiring the model's underlying knowledge base, but rather just a slight, coordinated "steering" or rotation of its existing features.
Mathematically, this means that when you fine-tune a model, the necessary weight changes ($\Delta W = W_{\text{finetune}} - W_{\text{pretrained}}$) possess a very low intrinsic dimensionality. LoRA assumes that the optimal weight update matrix $\Delta W$ naturally possesses a very low rank, meaning the required changes can be highly compressed and expressed as combinations of a small number of basis vectors. Ultimately, this implies that task-specific adaptations do not require computationally expensive full-rank weight updates, as they can be effectively and accurately represented by a low-rank decomposition without losing expressive power.

Q3. Why are LoRA layers typically applied to attention projection layers rather than all linear layers?

Ans. In the Transformer architecture, most of the critical task-specific learning and context routing takes place in the attention projection layers ($W_q, W_k, W_v, W_o$). While applying LoRA to all linear layers (including the massive MLP layers) does not increase inference latency after the weights are merged prior to inference ($W_{\text{merged}} = W + AB$), it does drastically increase the training cost and the number of trainable parameters. Empirical results from the original LoRA paper show that targeting just the attention weights captures the necessary task adaptations while maintaining the highest performance-to-parameter-efficiency ratio, saving training VRAM and compute without sacrificing accuracy.

Q4. How does the choice of rank affect the expressive capacity of the LoRA adaptation?

Ans.The choice of rank ($r$) acts as an information bottleneck that determines the expressive capacity of the adaptation by controlling the internal dimensionality between the low-rank matrices A and B.
- Underfitting (Too Low): A lower rank restricts the model to learning only the most dominant patterns of change. While highly efficient, if the rank is too low, the matrices lack the capacity to capture nuanced, task-specific details and feature rotations, leading to underfitting.
- The Efficiency Plateau (Optimal): For downstream tasks similar to the pre-training data, the required changes naturally have a low intrinsic dimension. Once a sufficient rank is reached, the model captures these details and matches the performance of full fine-tuning. Increasing the rank further yields virtually no improvements.
- Unseen Tasks (High Rank Needed): If the downstream task is fundamentally different or highly complex, the intrinsic dimension of the required update increases. This requires a much higher rank (or full fine-tuning) to provide enough trainable parameters to learn and map the new representations.
- Empirical Proof (SVD): The LoRA paper validates this efficiency plateau using Singular Value Decomposition (SVD). Their subspace similarity analysis shows that the top singular directions of low-rank and high-rank adaptations overlap significantly, demonstrating a shared dominant subspace. This proves that a lower rank successfully captures the most critical learning directions without needing full expressive capacity.

Q5. Why might a LoRA-adapted model perform comparably to a fully fine-tuned model on some tasks?

Ans. A LoRA-adapted model performs comparably to a fully fine-tuned model because it explicitly targets the dimensions where learning actually occurs. This is driven by three main factors:
- Low Intrinsic Dimension & Approximation: Because the intrinsic dimension of many specific tasks is low, full fine-tuning often implicitly arrives at a weight update matrix ($\Delta W$) that is already low-rank. LoRA enforces this low-rank structure explicitly from the start. By doing so, the low-rank matrices capture all the critical details, allowing LoRA to approximate the same solution when the optimal update is low-rank.
- Empirical Proof (Subspace Similarity): As shown via Singular Value Decomposition (SVD) in the LoRA paper, the top singular vectors of a low-rank update and a full-rank update overlap significantly. This proves that the most important learning directions are virtually identical in both approaches.
- Prevention of Overfitting and Catastrophic Forgetting: When full fine-tuning utilizes all dimensions, the extra parameters do not meaningfully participate in learning the task. Instead, they often capture noise by memorizing the training data. Because LoRA restricts the rank and freezes the base weights, it acts as a powerful natural regularizer. This prevents the model from learning training noise and avoids catastrophic forgetting, allowing LoRA to achieve comparable or sometimes superior generalization, especially on smaller datasets.


Q6. Under what conditions could LoRA lead to worse performance than full fine-tuning?

Ans.LoRA can underperform full fine-tuning in scenarios where its foundational assumption that the required weight updates have a low intrinsic dimension fundamentally breaks down, or when its practical constraints become a bottleneck. This typically occurs under the following conditions:
- Massive Domain Shifts (Unseen Tasks): If the downstream task involves a completely unseen domain (e.g., shifting from natural language to raw genomic sequences or a language with a completely distinct syntax), the model lacks the pre-trained features for these domains. It cannot simply "steer" or rotate existing knowledge; it must build entirely new representations from scratch, which intrinsically requires a high-rank, high-capacity update.
- Deep Factual Rewriting (Adaptation Beyond Attention): To maximize parameter efficiency, standard LoRA is typically applied only to attention projection matrices. However, while significant factual knowledge is encoded in the massive MLP (Multi-Layer Perceptron) blocks, it is also distributed across the network. If a task requires overwriting or injecting vast amounts of new factual data, applying LoRA solely to the attention mechanisms will fail to capture these necessary changes compared to full fine-tuning across all layers.
- Massive Datasets with Complex Nuance: LoRA's restricted rank acts as an information bottleneck and a strong regularizer. While this prevents overfitting on small datasets, if you possess an enormous, high-quality dataset for a highly complex task, this bottleneck can restrict the model's expressive capacity. In these cases, LoRA causes the model to underfit, whereas full fine-tuning utilizes its unrestricted parameter capacity to absorb all the deep nuances of the massive dataset.
- Suboptimal Configuration and Rank Selection: Even for tasks with a low intrinsic dimension, LoRA introduces critical new hyperparameters. If the specified rank ($r$) is artificially bottlenecked (set too low to save compute), the model will lack the capacity to capture critical task-specific details. Furthermore, poor tuning of the scaling factor ($\alpha$) or applying LoRA to suboptimal weight matrices can severely degrade performance compared to standard full fine-tuning.
 
